# 01 -- Temporal (T): Tối ưu Stage 1 (When)

Notebook này chỉ tập trung cải thiện điểm **T** (Gaussian temporal similarity). `core` đã cung cấp baseline `run_inference_vlm` hiện tại (T ghi trong `eval_core['T']` ở cell cuối của core, tính đến bản 'CHỐT CUỐI CÙNG' NumPro + coarse-to-fine, đo được T=0.4385 trên tập calibration).

**Quy tắc khi làm việc ở đây:** chỉ định nghĩa hàm MỚI (ví dụ `stage1_full_scan_v3`), không sửa trực tiếp các hàm import từ core. Ở cuối notebook, luôn so sánh T mới với `eval_core['T'].mean()` trên đúng `diverse_videos`.

In [ ]:
# [SETUP] Tu dong lay core pipeline tu GitHub repo -- khong can mount repo thu cong,
# chay duoc ngay tren Kaggle/Colab (session moi, chua co gi tren may) LAN local co
# san repo (khong clone lai, tu tim thu muc goc).
import os, sys, subprocess

REPO_URL  = 'https://github.com/HoangDinhBui/zero-shot-cctv-accident.git'
REPO_NAME = 'zero-shot-cctv-accident'

def _find_or_clone_repo_root():
    cwd = os.getcwd()
    # Da dang chay tu trong repo (notebook nay o notebooks/, core/ o thu muc cha)
    for base in (cwd, os.path.dirname(cwd)):
        if os.path.exists(os.path.join(base, 'core', 'shared_pipeline.ipynb')):
            return os.path.abspath(base)
    # Chua co -- clone ve thu muc hien tai (Kaggle/Colab session moi)
    if not os.path.exists(REPO_NAME):
        print(f'[SETUP] Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-q', REPO_URL], check=True)
    return os.path.abspath(REPO_NAME)

REPO_ROOT = _find_or_clone_repo_root()
sys.path.append(os.path.join(REPO_ROOT, 'core'))
print(f'[SETUP] REPO_ROOT = {REPO_ROOT}')

CORE_NB_PATH = os.path.join(REPO_ROOT, 'core', 'shared_pipeline.ipynb')
print(f'[SETUP] Loading core pipeline: {CORE_NB_PATH}')

In [ ]:
# Nap toan bo core (data, model loading, 3-stage baseline, eval, diverse_videos).
# Cac cell pip-install / fix NVRTC / load Qwen3-VL da nam san trong core, se tu
# chay khi %run toi do -- khong can them cell cai thu vien rieng o day.
%run {CORE_NB_PATH}

## Lịch sử: ablation từng lớp của T cascade (tham khảo, không tự chạy được ở đây)

Cell gốc này đo T riêng biệt cho từng lớp của classical-anchor cascade (`predict_accident_time_ensemble` -> `stage1_full_scan` -> `stage2_time_refine`). Bản thân `predict_accident_time_ensemble` cần YOLO detector + `compute_object_size_series` (Section 6.2/6.5 trong notebook gốc) -- phần đó **không có trong `core/` nữa** vì bản 'CHỐT CUỐI CÙNG' (patch cuối trong core) đã bỏ hẳn classical anchor, không còn gọi hàm này. Nếu muốn chạy lại đúng thí nghiệm này, cần import thêm `legacy/classical_pipeline.ipynb` (có định nghĩa `predict_accident_time_ensemble`) trước khi chạy cell tương tự.

Kết quả đã đo được lúc đó (giữ lại làm tài liệu tham khảo):

```
t_stage1_raw      : T = 0.2386
t_classical       : T = 0.1705
t_anchored        : T = 0.2314
t_refined_final   : T = 0.2262
```

Kết luận từ bảng trên: `t_anchored` thua `t_stage1_raw` -- neo classical (z-score + OSD) đang kéo tụt điểm, không phải kéo lên. Đây là lý do bản 'CHỐT CUỐI CÙNG' trong core đã bỏ hẳn classical anchor và chỉ dùng thẳng Stage 1 (NumPro) + coarse-to-fine (kết quả: T=0.4385, cao hơn hẳn cả 4 số ở trên -- 2 thay đổi đó, không phải anchor, mới là nguồn tăng điểm thật).

### 7d-quater. Thu nghiem NumPro-style: danh so khung hinh thay vi burn giay thap phan

Dua tren "Number it: Temporal Grounding Videos like Flipping Manga" (CVPR 2025,
arXiv:2411.10332, github.com/yongliang-wu/NumPro) -- doi burn-in tu `t=8.55s`
(giay lien tuc) sang so thu tu khung ("Frame 7") va hoi model chon SO KHUNG thay
vi hoi truc tiep so giay. Khong sua ham goc (`sample_frames_stamped`,
`build_stage1_prompt`) -- viet ham song song de A/B tren dung 20 video
calibration, so voi t_stage1_raw hien tai (T=0.2386) truoc khi quyet dinh thay
the ban chinh.

In [68]:
# [DIAG] NumPro-style: burn frame-index thay vi giay, hoi model chon so khung
def sample_frames_numbered(video_path, target_fps, t_start=None, t_end=None,
                           limit=32, max_side=768):
    """Nhu sample_frames_stamped nhung burn SO THU TU KHUNG (1, 2, 3, ...) thay
    vi t=xx.xxs. Tra ve ([times_seconds], [PIL.Image]) -- times[i] la giay that
    cua khung so (i+1), dung de quy doi nguoc sau khi model tra loi so khung."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or n <= 0:
        cap.release()
        return [], []
    duration = n / fps
    t0 = 0.0 if t_start is None else max(0.0, t_start)
    t1 = duration if t_end is None else min(duration, t_end)
    if t1 <= t0:
        t1 = min(duration, t0 + 1.0)

    times = np.arange(t0, t1 + 1e-6, 1.0 / max(target_fps, 0.1))
    if len(times) > limit:
        times = times[np.linspace(0, len(times) - 1, limit).round().astype(int)]

    imgs, kept_times = [], []
    for idx, t in enumerate(times, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.clip(round(t * fps), 0, n - 1)))
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        if max(h, w) > max_side:
            sc = max_side / float(max(h, w))
            frame = cv2.resize(frame, (int(round(w * sc)), int(round(h * sc))),
                               interpolation=cv2.INTER_AREA)
        # NumPro: so lon, ro, vi tri co dinh -- nhu danh so trang manga
        cv2.rectangle(frame, (8, 8), (90, 56), (0, 0, 0), -1)
        cv2.putText(frame, str(idx), (16, 46), cv2.FONT_HERSHEY_SIMPLEX,
                    1.4, (255, 255, 255), 3, cv2.LINE_AA)
        imgs.append(PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        kept_times.append(float(t))
    cap.release()
    return kept_times, imgs


NUMBERED_TIME_PROMPT_TEMPLATE = (
    "These {n} sequential CCTV frames are numbered 1 to {n}, shown in order like "
    "manga panels. Find the FRAME NUMBER where vehicles first make contact, or a "
    "vehicle first hits an object. Do not choose the peak impact or aftermath -- "
    "the first moment of contact only.\n\n"
    'Output ONLY this JSON: {{"collision_frame": <integer 1-{n}>}}'
)


def stage1_numbered_time(video_path, duration):
    times, imgs = sample_frames_numbered(video_path, 2.0, 0.0, duration, limit=32)
    if not imgs:
        return duration * 0.35
    prompt = NUMBERED_TIME_PROMPT_TEMPLATE.format(n=len(imgs))
    raw = vlm_generate_oom_safe(prompt, list(zip(times, imgs)), max_new_tokens=32, min_frames=4)
    parsed = _extract_json(raw)
    if not parsed or 'collision_frame' not in parsed:
        return duration * 0.35
    idx = int(np.clip(_safe_float(parsed['collision_frame'], 1), 1, len(times)))
    return times[idx - 1]


rows_numpro = []
for vp, gt_t in zip(diverse_videos, diverse_labels_df['accident_time']):
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    t_numpro = stage1_numbered_time(vp, duration)
    rows_numpro.append({'video': vp.name, 'gt': gt_t, 't_numpro': t_numpro})
    print(f"[STATUS] {vp.name[:35]:35s} gt={gt_t:6.2f} numpro={t_numpro:6.2f}")

numpro_df = pd.DataFrame(rows_numpro)
t_scores = [temporal_score(p, g) for p, g in zip(numpro_df['t_numpro'], numpro_df['gt'])]
print(f"\n[VERDICT] NumPro-style (frame-number burn-in) T = {np.mean(t_scores):.4f}")
print(f"          t_stage1_raw hien tai (giay lien tuc)     T = 0.2386")
print(f"          constant                                   T = 0.3800")
print("\n[QUYET DINH] Neu T o day > 0.2386 ro ret -- doi han sang burn frame-so,")
print("giu nguyen JSON schema tra ve nhung doi 'accident_time' thanh 'collision_frame'")
print("+ 1 buoc quy doi, roi thu lai coarse-to-fine (stage2_time_refine) TREN BIEU")
print("DIEN MOI nay -- co the refine se het net-negative khi hoat dong tren so khung")
print("thay vi giay thap phan.")


[WARNING] OOM -- retrying with 13 frames
[STATUS] Town03_head-on_wet_48.mp4           gt=  3.50 numpro=  2.00
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town06_head-on_wet_06.mp4           gt= 17.50 numpro=  9.50
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town03_head-on_night_40.mp4         gt=  6.45 numpro=  2.50
[WARNING] OOM -- retrying with 12 frames
[STATUS] Town06_head-on_wet_01.mp4           gt=  7.25 numpro=  7.00
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town04_rear-end_sunset_13.mp4       gt=  4.75 numpro=  2.50
[WARNING] OOM -- retrying with 9 frames
[STATUS] Town04_rear-end_rain_09.mp4         gt=  4.30 numpro=  1.00
[WARNING] OOM -- retrying with 15 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town05_rear-end_rain_142.mp4        gt=  8.65 numpro=  2.00
[WARNING] OOM -- retrying with 14 frames
[WARNING] OOM -- retrying with

## Thử nghiệm tiếp theo

Thêm cell mới bên dưới, định nghĩa `stage1_full_scan_v3` (hoặc `stage2_time_refine_v3`), rồi copy pattern đánh giá ở cell `[EVAL] Baseline...` của core, đổi `run_inference_vlm` trong vòng lặp bằng bản override của Stage 1/2, giữ nguyên Stage 3 (`stage3_grounding`) và Stage 2 loại va chạm (`classify_type_cascade`) là bản core.

---
# 🔧 KHU VỰC DEV — Temporal (T)
### Người phụ trách T viết code TỪ ĐÂY XUỐNG DƯỚI. Không sửa gì ở trên (đó là core + lịch sử tham khảo).
---

**Việc cần làm:** định nghĩa hàm mới cho Stage 1 và/hoặc Stage 2-refine (ví dụ `stage1_full_scan_v3`, `stage2_time_refine_v2`), KHÔNG sửa `stage1_full_scan` / `stage2_time_refine_numbered` gốc import từ `core`. Cuối cùng, đánh giá trên `diverse_videos` và so `T` mới với `eval_core['T'].mean()`.

In [ ]:
# TODO: dinh nghia ham Stage 1 / Stage 2 moi cho T o day.
# Vi du khung san (xoa va viet lai theo huong ban chon):
#
# def stage1_full_scan_vNEXT(video_path, duration, scene_layout=None):
#     ...
#     return {'accident_time': ..., 'center_x': ..., 'center_y': ..., 'type': ..., '_parsed': True}


In [ ]:
# [EVAL] So sanh T moi vs baseline core -- CHAY CELL NAY TRUOC KHI MO PR.
# Doi ten ham trong vong lap ben duoi thanh ham ban vua dinh nghia o tren.

rows_T_new = []
for vp, gt_t in zip(diverse_videos, diverse_labels_df['accident_time']):
    sub_path = 'videos/' + vp.name
    scene = SCENE_BY_PATH.get(sub_path)
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    # <-- doi 2 dong duoi day sang ham moi cua ban -->
    pred = stage1_full_scan(vp, duration, scene)
    t_final = stage2_time_refine_numbered(vp, pred['accident_time'], duration)

    pred['type'] = classify_type_cascade(vp, t_final, pred['type'])   # giu nguyen (core)
    pt = stage3_grounding(vp, t_final)                                 # giu nguyen (core)
    if pt is not None:
        pred['center_x'], pred['center_y'] = pt

    rows_T_new.append({'path': str(vp), 'accident_time': t_final,
                       'center_x': pred['center_x'], 'center_y': pred['center_y'],
                       'type': pred['type']})

eval_T_new = score_predictions(pd.DataFrame(rows_T_new), diverse_labels_df)
print(f"[BASELINE] T = {eval_core['T'].mean():.4f}")
print(f"[MOI]      T = {eval_T_new['T'].mean():.4f}")
print(f"[DELTA]        {eval_T_new['T'].mean() - eval_core['T'].mean():+.4f}")